# Configuration

In [1]:
from pathlib import Path
import sys

# Notebook-safe project path. In notebooks, __file__ is not available.
cwd = Path.cwd()
PROJECT_DIR = cwd.parent if cwd.name in {"pipeline", "notebooks"} else cwd
HELPER_DIR = PROJECT_DIR / "helper"

if str(HELPER_DIR) not in sys.path:
    sys.path.append(str(HELPER_DIR))

DATASET_PATH = PROJECT_DIR / "dataset" / "posture_data.csv"
REPORT_DIR = PROJECT_DIR / "reports"
MODEL_DIR = PROJECT_DIR / "models"

SENSOR1_COLS = ["Ax1", "Ay1", "Az1"]
SENSOR2_COLS = ["Ax2", "Ay2", "Az2"]
ACC_COLS = SENSOR1_COLS + SENSOR2_COLS
LABEL_COL = "label"

# For 10 Hz data, set SAMPLE_RATE_HZ = 10 and WINDOW_SIZE = 10.
SAMPLE_RATE_HZ = 10
LOWPASS_CUTOFF_HZ = 4.5
WINDOW_SIZE = 10
WINDOW_STEP = 10

LABEL_NAMES = {
    0: "Standing",
    1: "Sitting",
    2: "Sleeping",
    3: "Running",
    4: "Forward bending",
    5: "Backward bending",
}

dataset_path = DATASET_PATH
dataset_path

WindowsPath('d:/MonHoc_HK6/Do_an_2/Do_An/Detect_posture/dataset/posture_data.csv')

# Procesing Data

In [2]:
data_path = DATASET_PATH
output_path = REPORT_DIR / "processed_data.csv"

In [3]:
import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

EPS = 1e-8

def clean_dataset(df: pd.DataFrame, outlier_mode: str = "keep") -> pd.DataFrame:
    # Keep only valid numeric accelerometer values and integer labels.
    clean = df.copy()
    clean = clean.dropna(subset=ACC_COLS + [LABEL_COL])

    for col in ACC_COLS:
        clean[col] = pd.to_numeric(clean[col], errors="coerce")
    clean[LABEL_COL] = pd.to_numeric(clean[LABEL_COL], errors="coerce")
    clean = clean.dropna(subset=ACC_COLS + [LABEL_COL])
    clean[LABEL_COL] = clean[LABEL_COL].astype(int)

    if outlier_mode == "clip":
        # Clip outliers to the IQR boundary instead of removing rows.
        for col in ACC_COLS:
            q1 = clean[col].quantile(0.25)
            q3 = clean[col].quantile(0.75)
            iqr = q3 - q1
            clean[col] = clean[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    elif outlier_mode == "drop":
        # Drop rows that are outside the IQR boundary on any axis.
        mask = np.ones(len(clean), dtype=bool)
        for col in ACC_COLS:
            q1 = clean[col].quantile(0.25)
            q3 = clean[col].quantile(0.75)
            iqr = q3 - q1
            mask &= clean[col].between(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
        clean = clean.loc[mask]

    return clean.reset_index(drop=True)

def apply_lowpass(
    df: pd.DataFrame,
    fs: float = SAMPLE_RATE_HZ,
    cutoff: float = LOWPASS_CUTOFF_HZ,
    order: int = 2,
) -> pd.DataFrame:
    # Apply a light low-pass filter to reduce high-frequency noise.
    filtered = df.copy()
    filtered[ACC_COLS] = filtered[ACC_COLS].astype(float)
    nyquist = 0.5 * fs
    normalized_cutoff = cutoff / nyquist
    if not 0 < normalized_cutoff < 1:
        raise ValueError("cutoff must be lower than Nyquist frequency")

    b, a = butter(order, normalized_cutoff, btype="low")
    for label, group in filtered.groupby(LABEL_COL, sort=False):
        # Filter each posture block separately to avoid mixing labels.
        idx = group.index
        if len(group) <= max(len(a), len(b)) * 3:
            continue
        filtered.loc[idx, ACC_COLS] = filtfilt(b, a, group[ACC_COLS].to_numpy(), axis=0)
    return filtered


def add_instant_features(df: pd.DataFrame) -> pd.DataFrame:
    # Build per-sample features before aggregating them into windows.
    out = df.copy()
    for prefix, cols in [("s1", SENSOR1_COLS), ("s2", SENSOR2_COLS)]:
        ax, ay, az = [out[col] for col in cols]
        # Magnitude and normalized axes describe the gravity direction.
        mag = np.sqrt(ax**2 + ay**2 + az**2)
        out[f"{prefix}_acc_mag"] = mag
        out[f"{prefix}_Ax_norm"] = ax / (mag + EPS)
        out[f"{prefix}_Ay_norm"] = ay / (mag + EPS)
        out[f"{prefix}_Az_norm"] = az / (mag + EPS)
        # Roll and pitch approximate sensor orientation from acceleration.
        out[f"{prefix}_roll"] = np.arctan2(ay, az)
        out[f"{prefix}_pitch"] = np.arctan2(-ax, np.sqrt(ay**2 + az**2))
        out[f"{prefix}_abs_Ax_Ay"] = np.abs(ax - ay)
        out[f"{prefix}_abs_Ay_Az"] = np.abs(ay - az)
        out[f"{prefix}_abs_Az_Ax"] = np.abs(az - ax)

    v1 = out[SENSOR1_COLS].to_numpy()
    v2 = out[SENSOR2_COLS].to_numpy()
    # Angle between the two sensor vectors captures body segment posture.
    dot = np.sum(v1 * v2, axis=1)
    mag1 = out["s1_acc_mag"].to_numpy()
    mag2 = out["s2_acc_mag"].to_numpy()
    cos_angle = dot / (mag1 * mag2 + EPS)
    out["sensor_angle"] = np.arccos(np.clip(cos_angle, -1.0, 1.0))

    out["pitch_diff"] = out["s1_pitch"] - out["s2_pitch"]
    out["roll_diff"] = out["s1_roll"] - out["s2_roll"]
    out["acc_mag_diff"] = out["s1_acc_mag"] - out["s2_acc_mag"]

    for prefix, cols in [("s1", SENSOR1_COLS), ("s2", SENSOR2_COLS)]:
        # Jerk measures how much acceleration changes between samples.
        jerk = out.groupby(LABEL_COL, sort=False)[cols].diff().fillna(0.0)
        out[f"{prefix}_jerk_mag"] = np.sqrt(np.sum(jerk.to_numpy() ** 2, axis=1))

    return out


def _window_label(labels: pd.Series) -> int | None:
    # Use the only label in a clean window; skip mixed-label windows.
    modes = labels.mode()
    if len(modes) != 1:
        return None
    return int(modes.iloc[0])


def extract_window_features(
    df: pd.DataFrame,
    window_size: int = WINDOW_SIZE,
    step: int = WINDOW_STEP,
) -> pd.DataFrame:
    # Convert consecutive samples into one feature row per time window.
    enriched = add_instant_features(df)
    feature_cols = [col for col in enriched.columns if col != LABEL_COL]
    rows = []

    for label, group in enriched.groupby(LABEL_COL, sort=False):
        group = group.reset_index(drop=True)
        for start in range(0, len(group) - window_size + 1, step):
            window = group.iloc[start : start + window_size]
            window_label = _window_label(window[LABEL_COL])
            if window_label is None:
                continue

            row = {LABEL_COL: window_label}
            for col in feature_cols:
                # Basic window statistics work well for Logistic Regression.
                values = window[col].to_numpy(dtype=float)
                row[f"{col}_mean"] = float(np.mean(values))
                row[f"{col}_std"] = float(np.std(values, ddof=0))
                row[f"{col}_min"] = float(np.min(values))
                row[f"{col}_max"] = float(np.max(values))
                row[f"{col}_range"] = float(np.max(values) - np.min(values))

            # Keep extra mean/std features requested for orientation signals.
            for col in [
                "s1_Ax_norm",
                "s1_Ay_norm",
                "s1_Az_norm",
                "s2_Ax_norm",
                "s2_Ay_norm",
                "s2_Az_norm",
                "s1_roll",
                "s1_pitch",
                "s2_roll",
                "s2_pitch",
                "pitch_diff",
                "roll_diff",
            ]:
                values = window[col].to_numpy(dtype=float)
                row[f"{col}_mean_only"] = float(np.mean(values))
                row[f"{col}_std_only"] = float(np.std(values, ddof=0))

            for prefix in ["s1", "s2"]:
                # Jerk uses mean/std/max to summarize motion intensity.
                jerk = window[f"{prefix}_jerk_mag"].to_numpy(dtype=float)
                row[f"{prefix}_jerk_mag_mean_custom"] = float(np.mean(jerk))
                row[f"{prefix}_jerk_mag_std_custom"] = float(np.std(jerk, ddof=0))
                row[f"{prefix}_jerk_mag_max_custom"] = float(np.max(jerk))

            rows.append(row)

    return pd.DataFrame(rows)

In [4]:
data_df = clean_dataset(pd.read_csv(data_path))
filtered_data_df = apply_lowpass(data_df, fs=SAMPLE_RATE_HZ, cutoff=LOWPASS_CUTOFF_HZ)
features = extract_window_features(filtered_data_df, window_size=WINDOW_SIZE, step=WINDOW_STEP)
output = Path(output_path)
output.parent.mkdir(parents=True, exist_ok=True)
features.to_csv(output, index=False)
print(f"Saved features: {output} ({features.shape[0]} rows, {features.shape[1]} columns)")

Saved features: d:\MonHoc_HK6\Do_an_2\Do_An\Detect_posture\reports\processed_data.csv (4557 rows, 181 columns)


# Data Validation

In [5]:
def validate_dataset(df: pd.DataFrame) -> dict:
    # Check required columns and basic dataset health.
    required_cols = ACC_COLS + [LABEL_COL]
    missing_cols = [col for col in required_cols if col not in df.columns]

    numeric_df = df[[col for col in ACC_COLS if col in df.columns]]
    missing_values = df.isna().sum().to_dict()
    duplicated_rows = int(df.duplicated().sum())
    label_counts = df[LABEL_COL].value_counts().sort_index().to_dict() if LABEL_COL in df else {}

    outlier_summary = {}
    for col in numeric_df.columns:
        # Use the IQR rule to estimate possible outliers per axis.
        q1 = numeric_df[col].quantile(0.25)
        q3 = numeric_df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        mask = (numeric_df[col] < lower) | (numeric_df[col] > upper)
        outlier_summary[col] = {
            "count": int(mask.sum()),
            "ratio": float(mask.mean()),
            "lower": float(lower),
            "upper": float(upper),
        }

    return {
        "n_rows": int(len(df)),
        "n_cols": int(df.shape[1]),
        "missing_required_cols": missing_cols,
        "missing_values": missing_values,
        "duplicated_rows": duplicated_rows,
        "n_classes": int(df[LABEL_COL].nunique()) if LABEL_COL in df else 0,
        "label_counts": label_counts,
        "outliers_iqr": outlier_summary,
    }

validation_report = validate_dataset(data_df)

print("Validation summary")
print(f"- rows: {validation_report['n_rows']}")
print(f"- cols: {validation_report['n_cols']}")
print(f"- classes: {validation_report['n_classes']}")
print(f"- label_counts: {validation_report['label_counts']}")
print(f"- missing_required_cols: {validation_report['missing_required_cols']}")
print(f"- duplicated_rows: {validation_report['duplicated_rows']}")

print("\nMissing values")
for col, count in validation_report["missing_values"].items():
    print(f"- {col}: {count}")

print("\nOutliers by IQR")
for col, values in validation_report["outliers_iqr"].items():
    print(
        f"- {col}: count={values['count']}, "
        f"ratio={values['ratio']:.4f}, "
        f"lower={values['lower']:.2f}, "
        f"upper={values['upper']:.2f}"
    )

Validation summary
- rows: 45599
- cols: 7
- classes: 6
- label_counts: {0: 8274, 1: 7706, 2: 8472, 3: 5278, 4: 8208, 5: 7661}
- missing_required_cols: []
- duplicated_rows: 0

Missing values
- Ax1: 0
- Ay1: 0
- Az1: 0
- Ax2: 0
- Ay2: 0
- Az2: 0
- label: 0

Outliers by IQR
- Ax1: count=39, ratio=0.0009, lower=-14782.00, upper=34770.00
- Ay1: count=15, ratio=0.0003, lower=-30318.00, upper=38594.00
- Az1: count=6285, ratio=0.1378, lower=-456.00, upper=184.00
- Ax2: count=6036, ratio=0.1324, lower=-440.00, upper=440.00
- Ay2: count=6046, ratio=0.1326, lower=-259.00, upper=333.00
- Az2: count=0, ratio=0.0000, lower=-734.00, upper=2370.00


# Model Validation

In [6]:
import pickle

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def load_features(path: str | Path) -> tuple[pd.DataFrame, pd.Series]:
    # Split the feature table into input features and labels.
    feature_df = pd.read_csv(path)
    X = feature_df.drop(columns=[LABEL_COL])
    y = feature_df[LABEL_COL].astype(int)
    return X, y


def block_train_test_split(
    X: pd.DataFrame,
    y: pd.Series,
    test_size: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    # Split each label block by time order instead of random shuffling.
    train_parts = []
    test_parts = []

    data = X.copy()
    data[LABEL_COL] = y.to_numpy()

    for label, group in data.groupby(LABEL_COL, sort=False):
        split_idx = int(len(group) * (1 - test_size))
        if split_idx <= 0 or split_idx >= len(group):
            raise ValueError(f"Class {label} is too small for block split")
        train_parts.append(group.iloc[:split_idx])
        test_parts.append(group.iloc[split_idx:])

    train_df = pd.concat(train_parts, axis=0)
    test_df = pd.concat(test_parts, axis=0)

    X_train = train_df.drop(columns=[LABEL_COL])
    y_train = train_df[LABEL_COL].astype(int)
    X_test = test_df.drop(columns=[LABEL_COL])
    y_test = test_df[LABEL_COL].astype(int)
    return X_train, X_test, y_train, y_test


def make_train_test_split(
    X: pd.DataFrame,
    y: pd.Series,
    split_strategy: str = "random",
    test_size: float = 0.2,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    if split_strategy == "random":
        return train_test_split(
            X,
            y,
            test_size=test_size,
            random_state=42,
            stratify=y,
        )
    if split_strategy == "block":
        return block_train_test_split(X, y, test_size=test_size)
    raise ValueError("split_strategy must be 'random' or 'block'")


def tune_hyperparameters(
    output_path: str | Path,
    split_strategy: str = "random",
    test_size: float = 0.2,
) -> dict:
    # Tune hyperparameters only in this validation step.
    X, y = load_features(output_path)
    X_train, X_test, y_train, y_test = make_train_test_split(
        X,
        y,
        split_strategy=split_strategy,
        test_size=test_size,
    )

    pipe = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=3000,
                ),
            ),
        ]
    )

    search = GridSearchCV(
        pipe,
        param_grid={"model__C": [0.01, 0.1, 1.0, 10.0]},
        scoring="f1_macro",
        cv=5,
        n_jobs=1,
    )
    search.fit(X_train, y_train)

    return {
        "split_strategy": split_strategy,
        "best_C": search.best_params_["model__C"],
        "best_cv_f1_macro": search.best_score_,
        "train_rows": len(X_train),
        "test_rows": len(X_test),
        "train_label_counts": y_train.value_counts().sort_index().to_dict(),
        "test_label_counts": y_test.value_counts().sort_index().to_dict(),
    }

In [7]:
random_validation = tune_hyperparameters(output_path, split_strategy="random", test_size=0.2)
block_validation = tune_hyperparameters(output_path, split_strategy="block", test_size=0.2)

validation_results = {
    "random": random_validation,
    "block": block_validation,
}

validation_df = pd.DataFrame(
    [
        {
            "split_strategy": result["split_strategy"],
            "best_C": result["best_C"],
            "best_cv_f1_macro": result["best_cv_f1_macro"],
            "train_rows": result["train_rows"],
            "test_rows": result["test_rows"],
        }
        for result in validation_results.values()
    ]
)

display(validation_df)

,split_strategy,best_C,best_cv_f1_macro,train_rows,test_rows
0,random,10.0,0.994657,3645,912
1,block,10.0,0.987844,3643,914


# Model Training and Evaluation

In [8]:
def train_logistic_regression(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    C: float = 1.0,
):
    pipe = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=C,
                    class_weight="balanced",
                    solver="lbfgs",
                    max_iter=3000,
                ),
            ),
        ]
    )
    return pipe.fit(X_train, y_train)

def evaluate(model, X_test: pd.DataFrame, y_test: pd.Series) -> dict:
    # Report both global accuracy and class-balanced macro F1.
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "classification_report": classification_report(y_test, y_pred),
        "confusion_matrix": confusion_matrix(y_test, y_pred),
    }


def run_training(
    output_path: str | Path,
    split_strategy: str = "random",
    test_size: float = 0.2,
    C: float = 1.0,
    validation_score: float | None = None,
) -> dict:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)

    X, y = load_features(output_path)
    X_train, X_test, y_train, y_test = make_train_test_split(
        X,
        y,
        split_strategy=split_strategy,
        test_size=test_size,
    )

    model = train_logistic_regression(X_train, y_train, C=C)
    metrics = evaluate(model, X_test, y_test)

    # Save results 
    model_path = MODEL_DIR / f"logistic_regression_{split_strategy}_pipeline.pkl"
    with model_path.open("wb") as f:
        pickle.dump(model, f)

    report_path = REPORT_DIR / f"training_report_{split_strategy}.txt"
    with report_path.open("w", encoding="utf-8") as f:
        f.write(f"split_strategy: {split_strategy}\n")
        f.write(f"test_size: {test_size}\n")
        f.write(f"selected_C: {C}\n")
        if validation_score is not None:
            f.write(f"validation_cv_f1_macro: {validation_score:.6f}\n")
        f.write(f"train_rows: {len(X_train)}\n")
        f.write(f"test_rows: {len(X_test)}\n")
        f.write(f"train_label_counts: {y_train.value_counts().sort_index().to_dict()}\n")
        f.write(f"test_label_counts: {y_test.value_counts().sort_index().to_dict()}\n\n")
        f.write(f"accuracy: {metrics['accuracy']:.6f}\n")
        f.write(f"f1_macro: {metrics['f1_macro']:.6f}\n\n")
        f.write("classification_report:\n")
        f.write(metrics["classification_report"])
        f.write("\nconfusion_matrix:\n")
        f.write(str(metrics["confusion_matrix"]))

    return {
        "split_strategy": split_strategy,
        "selected_C": C,
        "validation_cv_f1_macro": validation_score,
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "model_path": model_path,
        "report_path": report_path,
        "classification_report": metrics["classification_report"],
        "confusion_matrix": metrics["confusion_matrix"],
    }


In [9]:
random_result = run_training(
    output_path,
    split_strategy="random",
    test_size=0.2,
    C=validation_results["random"]["best_C"],
    validation_score=validation_results["random"]["best_cv_f1_macro"],
)
block_result = run_training(
    output_path,
    split_strategy="block",
    test_size=0.2,
    C=validation_results["block"]["best_C"],
    validation_score=validation_results["block"]["best_cv_f1_macro"],
)

results_df = pd.DataFrame(
    [
        {
            "split_strategy": random_result["split_strategy"],
            "selected_C": random_result["selected_C"],
            "validation_cv_f1_macro": random_result["validation_cv_f1_macro"],
            "accuracy": random_result["accuracy"],
            "f1_macro": random_result["f1_macro"],
            "model_path": str(random_result["model_path"]),
            "report_path": str(random_result["report_path"]),
        },
        {
            "split_strategy": block_result["split_strategy"],
            "selected_C": block_result["selected_C"],
            "validation_cv_f1_macro": block_result["validation_cv_f1_macro"],
            "accuracy": block_result["accuracy"],
            "f1_macro": block_result["f1_macro"],
            "model_path": str(block_result["model_path"]),
            "report_path": str(block_result["report_path"]),
        },
    ]
)
display(results_df)
print("Random split classification report:")
print(random_result["classification_report"])
print("Block split classification report:")
print(block_result["classification_report"])

,split_strategy,selected_C,validation_cv_f1_macro,accuracy,f1_macro,model_path,report_path
0,random,10.0,0.994657,1.000000,1.00000,d:\MonHoc_HK6\Do_an_2\Do_An\Detect_posture\mod...,d:\MonHoc_HK6\Do_an_2\Do_An\Detect_posture\rep...
1,block,10.0,0.987844,0.998906,0.99899,d:\MonHoc_HK6\Do_an_2\Do_An\Detect_posture\mod...,d:\MonHoc_HK6\Do_an_2\Do_An\Detect_posture\rep...


Random split classification report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       166
           1       1.00      1.00      1.00       154
           2       1.00      1.00      1.00       170
           3       1.00      1.00      1.00       105
           4       1.00      1.00      1.00       164
           5       1.00      1.00      1.00       153

    accuracy                           1.00       912
   macro avg       1.00      1.00      1.00       912
weighted avg       1.00      1.00      1.00       912

Block split classification report:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       166
           1       1.00      1.00      1.00       154
           2       1.00      1.00      1.00       170
           3       1.00      1.00      1.00       106
           4       0.99      1.00      1.00       164
           5       1.00      1.00      1.00       154

    ac